In [ ]:
# [1/7] Mount Google Drive & Verify Project Structure
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/code'
sys.path.insert(0, PROJECT_ROOT)

REQUIRED = ['data/train.json', 'data/val.json']
print('Project file check:')
all_ok = True
for f in REQUIRED:
    p = os.path.join(PROJECT_ROOT, f)
    ok = os.path.isfile(p)
    if not ok:
        all_ok = False
    print(f'  {"OK" if ok else "MISSING":>7s}  {f}')

if not all_ok:
    raise FileNotFoundError('Missing required files — run prepare_data.py first')
print(f'\nPROJECT_ROOT = {PROJECT_ROOT}')

In [ ]:
# [2/7] Install Dependencies & HF Login
!pip install -q transformers peft trl datasets accelerate bitsandbytes "torchao>=0.16.0" 2>/dev/null

import torch, transformers, peft
print(f'torch={torch.__version__}')
print(f'transformers={transformers.__version__}')
print(f'peft={peft.__version__}')

# ---- HuggingFace Login (required for gated Qwen3-8B) ----
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print(f"  HF token loaded from Colab secrets.")
except Exception:
    hf_token = None
    print("  Colab secrets not available, falling back to interactive login...")

if hf_token:
    login(token=hf_token)
else:
    print("  Enter your HF token (get it at https://huggingface.co/settings/tokens):")
    login()

print("  Authenticated OK.")

In [ ]:
# [3/7] Setup: imports + paths + config
import json, os, sys
from pathlib import Path
from datetime import datetime

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
)
from peft import LoraConfig
from trl import SFTTrainer
from datasets import Dataset, concatenate_datasets

# ---- Paths ----
DATA_DIR = Path(PROJECT_ROOT) / "data"
TRAIN_PATH = DATA_DIR / "train.json"
VAL_PATH = DATA_DIR / "val.json"
MODEL_DIR = Path(PROJECT_ROOT) / "models"
OUTPUT_DIR = MODEL_DIR / "Qwen8b_finetuned"
CHECKPOINT_DIR = MODEL_DIR / "Qwen8b_checkpoints"

MODEL_DIR.mkdir(exist_ok=True)

# ---- Model ----
BASE_MODEL = "Qwen/Qwen3-8B"

# ---- LoRA hyperparams ----
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ---- Training hyperparams ----
NUM_EPOCHS = 12
LEARNING_RATE = 1e-5
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
LR_SCHEDULER_TYPE = "cosine"
MAX_SEQ_LENGTH = 512
EARLY_STOPPING_PATIENCE = 4
SAVE_LIMIT = 3

USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8

# ---- LoRAFT system prompt (NO predicate list) ----
LORAFT_SYSTEM_PROMPT = (
    "You are a helpful AI assistant that translates Natural Language (NL) text in "
    "First-Order Logic (FOL) using only the given quantors and junctors:\n"
    "∀ (for all), ∃ (there exists), ¬ (not), ∧ (and), "
    "∨ (or), → (implies), ↔ (if and only if), ⊕ (xor).\n"
    "Start your answer with '\U0001D719=' followed by the FOL-formula. "
    "Do not include any other text."
)

print(f"{'='*55}")
print(f"  LoRAFT-STYLE FINE-TUNING (Qwen8b)")
print(f"{'='*55}")
print(f"  Base model:     {BASE_MODEL}")
print(f"  Train data:     train.json (18k)")
print(f"  Val data:       val.json")
print(f"  Output:         {OUTPUT_DIR}")
print(f"  LoRA:           r={LORA_R}  alpha={LORA_ALPHA}  dropout={LORA_DROPOUT}")
print(f"  Prompt format:  LoRAFT decoder-only (NO Predicates=[...])")
print(f"  Epochs:         {NUM_EPOCHS}")
print(f"  LR:             {LEARNING_RATE}")
print(f"  Batch size:     {PER_DEVICE_BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} = {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS} eff")
print(f"  Precision:      {'bf16' if USE_BF16 else 'fp16'}")
print(f"{'='*55}")

In [ ]:
# [4/7] Load & reformat data with LoRAFT-style prompts (no predicate list)
print("Loading datasets...")

with open(TRAIN_PATH, encoding="utf-8") as f:
    train_raw = json.load(f)
with open(VAL_PATH, encoding="utf-8") as f:
    val_raw = json.load(f)

print(f"  train: {len(train_raw)} samples")
print(f"  val:    {len(val_raw)} samples")

# ---- Reformat: NL+FOL → messages format with LoRAFT prompt ----
def formatting_func(example):
    return {"messages": [
        {"role": "system", "content": LORAFT_SYSTEM_PROMPT},
        {"role": "user", "content": example["NL"]},
        {"role": "assistant", "content": f"𝜙={example['FOL']}"},
    ]}

train_dataset = (
    Dataset.from_list(train_raw)
    .map(formatting_func, remove_columns=["NL", "FOL"], batched=False)
)
val_dataset = (
    Dataset.from_list(val_raw)
    .map(formatting_func, remove_columns=["NL", "FOL"], batched=False)
)

print(f"\n  Formatted train: {len(train_dataset)} samples")
print(f"  Formatted val:   {len(val_dataset)} samples")

# ---- Quick format check ----
sample = train_dataset[0]
msgs = sample["messages"]
print(f"\n  Sample format check:")
print(f"    system:    {msgs[0]['content'][:120]}...")
print(f"    user:      {msgs[1]['content'][:80]}...")
print(f"    assistant: {msgs[2]['content'][:120]}...")
print(f"\n  (LoRAFT format, 𝜙= prefix, NO predicate list — same prompt for all samples)")

In [ ]:
# [5/7] Load model + apply LoRA
print(f"Loading base model: {BASE_MODEL}...")
print("  (this may take 3-8 minutes)")

# ---- Tokenizer ----
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ---- Model ----
load_kwargs = {
    "trust_remote_code": True,
    "use_cache": False,
    "device_map": "auto",
}

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb >= 40:
    print(f"  VRAM: {vram_gb:.1f} GB → using bf16 full precision")
    load_kwargs["torch_dtype"] = torch.bfloat16
else:
    print(f"  VRAM: {vram_gb:.1f} GB → using 4-bit quantization")
    from transformers import BitsAndBytesConfig
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **load_kwargs)

# ---- LoRA Config ----
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

# ---- Verify target modules ----
all_modules = {n for n, _ in model.named_modules()}
found = [m for m in LORA_TARGET_MODULES if any(n.endswith(m) for n in all_modules)]
missing = [m for m in LORA_TARGET_MODULES if m not in found]
print(f"\n  LoRA target modules found: {found}")
if missing:
    print(f"  WARNING — modules NOT found: {missing}")
    LORA_TARGET_MODULES = [m for m in LORA_TARGET_MODULES if m in found]

print(f"\n  Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"  Total params:     {sum(p.numel() for p in model.parameters()):,}")
pct = 100 * sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters())
print(f"  LoRA ratio:       {pct:.2f}%")

In [ ]:
# [6/7] Train + Save
print(f"{'='*55}")
print(f"  TRAINING")
print(f"{'='*55}")

num_steps_per_epoch = len(train_dataset) // (PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
total_steps = num_steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

print(f"  Train samples: {len(train_dataset)}")
print(f"  Val samples:   {len(val_dataset)}")
print(f"  Steps/epoch:   ~{num_steps_per_epoch}")
print(f"  Total steps:   ~{total_steps}  (warmup: {warmup_steps})")
print(f"  Checkpoints:   {CHECKPOINT_DIR}")
print(f"  Final model:   {OUTPUT_DIR}")
print()

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=SAVE_LIMIT,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_steps=warmup_steps,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=2,
    remove_unused_columns=False,
)


tokenizer.model_max_length = MAX_SEQ_LENGTH

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
    ],
)

# ---- Checkpoint resume ----
resume_from_ckpt = False
if CHECKPOINT_DIR.exists():
    ckpts = sorted(CHECKPOINT_DIR.glob("checkpoint-*"))
    if ckpts:
        resume_from_ckpt = True
        print(f"\n  *** Resuming from checkpoint: {ckpts[-1].name}")
        print(f"  *** {len(ckpts)} checkpoint(s) found in {CHECKPOINT_DIR}")
    else:
        print(f"\n  No checkpoint found — training from scratch")
else:
    print(f"\n  No checkpoint dir yet — training from scratch")

t_start = datetime.now()
trainer.train(resume_from_checkpoint=resume_from_ckpt)
t_end = datetime.now()

elapsed = (t_end - t_start).total_seconds() / 60
print(f"\nTraining complete in {elapsed:.1f} min")

# ---- Save LoRA adapter + tokenizer ----
print(f"\nSaving LoRA adapter + tokenizer to {OUTPUT_DIR}...")
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))


In [ ]:
# [7/7] Quick Sanity Check — generate one sample with the fine-tuned model
import torch, re
from peft import PeftModel

print("Loading fine-tuned model for sanity check...")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
base_model_check = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if vram_gb >= 40 else None,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer_check = AutoTokenizer.from_pretrained(str(OUTPUT_DIR), trust_remote_code=True)
if tokenizer_check.pad_token is None:
    tokenizer_check.pad_token = tokenizer_check.eos_token

model_check = PeftModel.from_pretrained(base_model_check, str(OUTPUT_DIR))
model_check.eval()

# Test NL
test_nl = "Fish swim."
messages = [
    {"role": "system", "content": LORAFT_SYSTEM_PROMPT},
    {"role": "user", "content": test_nl},
]
text = tokenizer_check.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer_check(text, return_tensors="pt").to(model_check.device)
input_len = inputs.input_ids.shape[1]

with torch.no_grad():
    outputs = model_check.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.0,
        do_sample=False,
        pad_token_id=tokenizer_check.pad_token_id,
        eos_token_id=tokenizer_check.eos_token_id,
    )

response = tokenizer_check.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
# Strip the training 𝜙= prefix
fol = re.sub(r'^[\U0001D719]\s*=\s*', '', response)

print(f"\n  Sanity check — NL:  {test_nl}")
print(f"  Raw output:         {response[:150]}")
print(f"  Cleaned FOL:        {fol[:150]}")
print(f"\n  Expected: ∀x (Fish(x) → Swims(x))")
print(f"\nModel saved to: {OUTPUT_DIR}")
print("Done. Ready for Qwen8b inference notebook.")